# Smoke Test - FEVER
10 examples, real LLM calls, no plots. Verifies end-to-end wiring (k=10, global corpus, injection) before full experiments.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

N_SMOKE     = 10
MODEL       = cfg["models"]["default"]
PROMPT_TYPE = cfg["prompts"]["default"]
TEMPERATURE = cfg["models"]["temperature"]

print(f"model={MODEL}  prompt={PROMPT_TYPE}  k={cfg['retrieval']['k']}  n_smoke={N_SMOKE}")

In [ ]:
from src.data.fever_loader import load_fever

examples = load_fever(
    "../" + cfg["dataset"]["fever_dev"],
    max_examples=N_SMOKE,
)

print(f"Loaded {len(examples)} examples")
print("Labels:", [e["label"] for e in examples])

In [ ]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever
from src.generation.llm_client import HuggingFaceClient

emb_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
embedder = Embedder(model_name=cfg["retrieval"]["embedding_model"], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg["retrieval"]["k"])

llm_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])
llm = HuggingFaceClient(model=MODEL, temperature=TEMPERATURE, cache_dir=llm_cache)
print(f"Embedder: {cfg['retrieval']['embedding_model']}  k={cfg['retrieval']['k']}  model={MODEL}")

In [ ]:
from src.data.fever_loader import load_fever
from src.evaluation.scorer import run as run_scorer

poisoned = load_fever("../" + cfg["dataset"]["fever_dev_poisoned"], max_examples=N_SMOKE)
print(f"Loaded {len(poisoned)} pre-computed poisoned examples")

with embedder, llm:
    # r=0: original gold passages
    metrics_clean = run_scorer(
        examples=examples,
        retriever=retriever,
        llm=llm,
        prompt_type=PROMPT_TYPE,
        seed=cfg["seed"],
    )
    # r=1: pre-computed LLM-negated passages (loaded from dev_poisoned.jsonl)
    metrics_poisoned = run_scorer(
        examples=poisoned,
        retriever=retriever,
        llm=llm,
        prompt_type=PROMPT_TYPE,
        seed=cfg["seed"],
    )

print("=== r=0 (clean) ===")
for k, v in metrics_clean.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n=== r=1 (poisoned) ===")
for k, v in metrics_poisoned.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nFEVER smoke passed." if metrics_clean and metrics_poisoned else "Check errors above.")